<a href="https://colab.research.google.com/github/iking919/Detecting_Financial_Fraud_via_GNNs/blob/Preprocessing/02_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Detecting Financial Fraud via Graph Neural Networks: A Multi-Dataset, Graph-Based Learning Approach

### Haoyuan Chen, Izaak King, Bryan Gelnett

# Data Exploration


## Dataset Preprocessing

In this notebook, we will build off of our Exploratory Data Analysis by processing and cleaning our selected datasets before we begin training models. This section will include structuring our data to fit our GNN models, as well as preliminary models for baseline comparisons.

<br>

Our datasets are:

[PaySim Mobile Money Dataset]( https://www.kaggle.com/datasets/ealaxi/paysim1)

[IEEE-CIS Fraud Detection Dataset](https://www.kaggle.com/datasets/lnasiri007/ieeecis-fraud-detection)

[Elliptic Data Set](https://www.kaggle.com/datasets/ellipticco/elliptic-data-set)

**Note**: Our datasets are very large and consume a lot of RAM when performing operations. We are careful to free RAM whenever possible, but sessions may crash.

First, we upload access the Kaggle API by uploading our `kaggle.json` file. This will allow us to easily download our datasets into our Colab session.

In [5]:
from google.colab import files

files.upload() # Upload kaggle.json file into Colab session

!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

Saving kaggle.json to kaggle.json


To maintain a clean repository structure, we will download all datasets into a dedicated `data/` directory. Each dataset will have its own subfolder.

In [6]:
import os

# Create a base directory for all data
DATA_DIR = "./data"
os.makedirs(DATA_DIR, exist_ok=True)
print(f"Data directory created at: {DATA_DIR}")

Data directory created at: ./data


### 1. PaySim Mobile Money Dataset
Link: https://www.kaggle.com/datasets/ealaxi/paysim1

In [7]:
!kaggle datasets download -d ealaxi/paysim1 -p {DATA_DIR}/paysim --unzip
print("PaySim dataset downloaded and unzipped successfully.")

Dataset URL: https://www.kaggle.com/datasets/ealaxi/paysim1
License(s): CC-BY-SA-4.0
100% 178M/178M [00:01<00:00, 109MB/s]

PaySim dataset downloaded and unzipped successfully.


### 2. IEEE-CIS Fraud Detection Dataset
Link: https://www.kaggle.com/datasets/lnasiri007/ieeecis-fraud-detection

<br>

**Note:**  
This dataset is already split into training and test sets as provided by the competition.

We will use the train_transaction + train_identity files for training our models, and the test_transaction + test_identity files for evaluation. This preserves the official split and ensures fair benchmarking of model performance.

In [8]:
!kaggle datasets download -d lnasiri007/ieeecis-fraud-detection -p {DATA_DIR}/ieee-cis --unzip
print("IEEE-CIS Fraud Detection dataset downloaded and unzipped successfully.")

Dataset URL: https://www.kaggle.com/datasets/lnasiri007/ieeecis-fraud-detection
License(s): unknown
100% 118M/118M [00:01<00:00, 77.2MB/s]

IEEE-CIS Fraud Detection dataset downloaded and unzipped successfully.


### 3. Elliptic Data Set
Link: https://www.kaggle.com/datasets/ellipticco/elliptic-data-set

In [9]:
!kaggle datasets download -d ellipticco/elliptic-data-set -p {DATA_DIR}/elliptic --unzip
print("Elliptic Data Set downloaded and unzipped successfully.")

Dataset URL: https://www.kaggle.com/datasets/ellipticco/elliptic-data-set
License(s): Attribution-NonCommercial-NoDerivatives 4.0 International (CC BY-NC-ND 4.0)
100% 146M/146M [00:01<00:00, 125MB/s]

Elliptic Data Set downloaded and unzipped successfully.


### Importing Libraries

In [5]:
import pandas as pd
import numpy as np
import os
import gc


from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

print("Libraries imported successfully.")

Libraries imported successfully.


## Load Datasets

We load the PaySim, IEEE-CIS, and Elliptic datasets into pandas DataFrames.


In [11]:
# PaySim
df_paysim = pd.read_csv('/content/data/paysim/PS_20174392719_1491204439457_log.csv')
print("PaySim Shape:", df_paysim.shape)
display(df_paysim.head())

# IEEE-CIS train + test
df_ieee_trans = pd.read_csv('/content/data/ieee-cis/train_transaction.csv')
df_ieee_id = pd.read_csv('/content/data/ieee-cis/train_identity.csv')
df_ieee_test_trans = pd.read_csv('/content/data/ieee-cis/test_transaction.csv')
df_ieee_test_id = pd.read_csv('/content/data/ieee-cis/test_identity.csv')

print("\nIEEE Train Transaction Shape:", df_ieee_trans.shape)
display(df_ieee_trans.head())
print("\nIEEE Train Identity Shape:", df_ieee_id.shape)
display(df_ieee_id.head())
print("\nIEEE Test Transaction Shape:", df_ieee_test_trans.shape)
display(df_ieee_test_trans.head())
print("\nIEEE Test Identity Shape:", df_ieee_test_id.shape)
display(df_ieee_test_id.head())

# Elliptic
df_elliptic_features = pd.read_csv('/content/data/elliptic/elliptic_bitcoin_dataset/elliptic_txs_features.csv', header=None)
df_elliptic_edges = pd.read_csv('/content/data/elliptic/elliptic_bitcoin_dataset/elliptic_txs_edgelist.csv')
df_elliptic_classes = pd.read_csv('/content/data/elliptic/elliptic_bitcoin_dataset/elliptic_txs_classes.csv')

print("\nElliptic Features Shape:", df_elliptic_features.shape)
display(df_elliptic_features.head())
print("\nElliptic Edges Shape:", df_elliptic_edges.shape)
display(df_elliptic_edges.head())
print("\nElliptic Classes Shape:", df_elliptic_classes.shape)
display(df_elliptic_classes.head())

PaySim Shape: (6362620, 11)


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0



IEEE Train Transaction Shape: (590540, 394)


,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,V330,V331,V332,V333,V334,V335,V336,V337,V338,V339
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.0,W,18132,567.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0



IEEE Train Identity Shape: (144233, 41)


...

,0,1,2,3,4,5,6,7,8,9,...,157,158,159,160,161,162,163,164,165,166
0,230425980,1,-0.171469,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.162097,...,-0.562153,-0.600999,1.461330,1.461369,0.018279,-0.087490,-0.131155,-0.097524,-0.120613,-0.119792
1,5530458,1,-0.171484,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.162112,...,0.947382,0.673103,-0.979074,-0.978556,0.018279,-0.087490,-0.131155,-0.097524,-0.120613,-0.119792
2,232022460,1,-0.172107,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.162749,...,0.670883,0.439728,-0.979074,-0.978556,-0.098889,-0.106715,-0.131155,-0.183671,-0.120613,-0.119792
3,232438397,1,0.163054,1.963790,-0.646376,12.409294,-0.063725,9.782742,12.414558,-0.163645,...,-0.577099,-0.613614,0.241128,0.241406,1.072793,0.085530,-0.131155,0.677799,-0.120613,-0.119792
4,230460314,1,1.011523,-0.081127,-1.201369,1.153668,0.333276,1.312656,-0.061584,-0.163523,...,-0.511871,-0.400422,0.517257,0.579382,0.018279,0.277775,0.326394,1.293750,0.178136,0.179117



Elliptic Edges Shape: (234355, 2)


,txId1,txId2
0,230425980,5530458
1,232022460,232438397
2,230460314,230459870
3,230333930,230595899
4,232013274,232029206



Elliptic Classes Shape: (203769, 2)


,txId,class
0,230425980,unknown
1,5530458,unknown
2,232022460,unknown
3,232438397,2
4,230460314,unknown


## PaySim Preprocessing and Graph Construction

The PaySim dataset contains synthetic mobile money transactions. Our goal is to prepare both tabular data for baseline ML and graph-structured data for GNNs.

We log-transform transaction amounts, encode transaction types numerically, and normalize selected numeric features. For the graph structure, we treat each transaction as a node and create edges whenever transactions share the same origin or destination account. This ensures that sequential flows and repeated patterns are captured in the graph.

In [14]:
# Feature Engineering & Encoding
df_paysim = df_paysim.copy()
df_paysim['log_amount'] = np.log1p(df_paysim['amount'])
df_paysim['type_encoded'] = LabelEncoder().fit_transform(df_paysim['type'])

# Separate features and target
features_p = ['log_amount','type_encoded','oldbalanceOrg','newbalanceOrig','oldbalanceDest','newbalanceDest','step']
X_paysim = df_paysim[features_p]
y_paysim = df_paysim['isFraud']

# Standardize features
scaler = StandardScaler()
X_paysim_scaled = scaler.fit_transform(X_paysim)

# Assign unique tx_id for graph construction
df_paysim['tx_id'] = df_paysim.index

# Graph Construction (Edges)

# Connect transactions with the same origin account
edges_orig = (
    df_paysim[['tx_id','nameOrig']]
    .merge(df_paysim[['tx_id','nameOrig']], on='nameOrig')
    .query('tx_id_x != tx_id_y')
    [['tx_id_x','tx_id_y']]
)
edges_orig.columns = ['source','target']

# Connect transactions with the same destination account
edges_dest = (
    df_paysim[['tx_id','nameDest']]
    .merge(df_paysim[['tx_id','nameDest']], on='nameDest')
    .query('tx_id_x != tx_id_y')
    [['tx_id_x','tx_id_y']]
)
edges_dest.columns = ['source','target']

# Combine edges
edges_paysim = pd.concat([edges_orig, edges_dest], ignore_index=True)

# Create Nodes DataFrame
nodes_paysim = pd.DataFrame(X_paysim_scaled, columns=features_p)
nodes_paysim['tx_id'] = df_paysim['tx_id']
nodes_paysim['label'] = y_paysim.values

# Final cleanup
del df_paysim
del X_paysim
del X_paysim_scaled
del edges_orig
del edges_dest
gc.collect()

print(f"Nodes shape: {nodes_paysim.shape}")
print(f"Edges shape: {edges_paysim.shape}")

Nodes shape: (6362620, 9)
Edges shape: (64867992, 2)


## IEEE-CIS Preprocessing and Graph Construction

The IEEE-CIS dataset is heterogeneous, with transactions linked to identity features (cards, emails, devices). We first merge transaction and identity tables, then handle missing data by filling numeric columns with medians and categorical columns with "unknown," followed by label encoding.

For baseline ML, tabular features are standardized. For GNN graph construction, each transaction is a node, and edges are created between transactions that share key attributes such as card1, P_emaildomain, or DeviceType.

In [12]:
# Combine train and test to create a unified graph
def preprocess_ieee(df_trans, df_id, is_train=True):
    df_ieee = df_trans.merge(df_id, on='TransactionID', how='left')
    del df_trans, df_id
    gc.collect()

    # Numeric columns
    num_cols = df_ieee.select_dtypes(include=np.number).columns
    for col in num_cols:
        df_ieee[col] = df_ieee[col].fillna(df_ieee[col].median())

    # Categorical columns
    cat_cols = df_ieee.select_dtypes(include='object').columns
    for col in cat_cols:
        df_ieee[col] = df_ieee[col].fillna('unknown')
        df_ieee[col] = LabelEncoder().fit_transform(df_ieee[col].astype(str))

    if is_train:
        y = df_ieee['isFraud'].values
        df_ieee.drop(columns=['TransactionID', 'isFraud'], inplace=True)
    else:
        y = None
        df_ieee.drop(columns=['TransactionID'], inplace=True)

    # Standardize
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(df_ieee)

    # Assign unique tx_id
    df_ieee['tx_id'] = np.arange(len(df_ieee))

    # Graph edges by key attributes
    edges_list = []
    for col in ['card1', 'P_emaildomain', 'DeviceType']:
        tmp = df_ieee[['tx_id', col]].sort_values(by=[col, 'tx_id'])
        tmp['next_tx_id'] = tmp['tx_id'].shift(-1)
        tmp['next_col'] = tmp[col].shift(-1)
        valid_edges = tmp[tmp[col] == tmp['next_col']]
        edge_df = valid_edges[['tx_id', 'next_tx_id']].rename(columns={'tx_id':'source','next_tx_id':'target'})
        edges_list.append(edge_df)

    edges = pd.concat(edges_list, ignore_index=True)
    edges['target'] = edges['target'].astype(int)

    nodes = pd.DataFrame(X_scaled, columns=df_ieee.columns.drop('tx_id'))
    nodes['tx_id'] = df_ieee['tx_id'].values
    if is_train:
        nodes['label'] = y

    del df_ieee, X_scaled, edges_list
    gc.collect()
    return nodes, edges

# Preprocess train and test separately
nodes_ieee_train, edges_ieee_train = preprocess_ieee(df_ieee_trans, df_ieee_id, is_train=True)
nodes_ieee_test, edges_ieee_test = preprocess_ieee(df_ieee_test_trans, df_ieee_test_id, is_train=False)

print(f"Train Nodes: {nodes_ieee_train.shape}, Train Edges: {edges_ieee_train.shape}")
print(f"Test Nodes: {nodes_ieee_test.shape}, Test Edges: {edges_ieee_test.shape}")

/tmp/ipykernel_24335/3440681472.py:30: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_ieee['tx_id'] = np.arange(len(df_ieee))
/tmp/ipykernel_24335/3440681472.py:30: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_ieee['tx_id'] = np.arange(len(df_ieee))


Train Nodes: (590540, 434), Train Edges: (1758004, 2)
Test Nodes: (506691, 433), Test Edges: (1506765, 2)


## Elliptic Preprocessing and Graph Construction

The Elliptic dataset is naturally graph-structured, with Bitcoin transactions as nodes and fund flows as edges. We rename features, merge labels, and remove unknown transactions. Node features include all precomputed features from the dataset, while edges are taken directly from the original edgelist. Each node contains a unique transaction ID, its features, and its label (licit vs illicit). This preserves the full graph structure for GNN training and evaluation.

In [13]:
# Rename columns for features DataFrame
num_cols = df_elliptic_features.shape[1]
feature_cols = [f'feature_{i}' for i in range(1,num_cols-1)]
df_elliptic_features.columns = ['txId','timestep'] + feature_cols

# Merge features with labels and filter out unknown classes
df_elliptic = df_elliptic_features.merge(df_elliptic_classes, on='txId')
df_elliptic = df_elliptic[df_elliptic['class'] != 'unknown']
df_elliptic['class'] = df_elliptic['class'].astype(int)

# Separate features and target
X_elliptic = df_elliptic.drop(columns=['txId','class'])
y_elliptic = df_elliptic['class']

# Standardize features
scaler = StandardScaler()
X_elliptic_scaled = scaler.fit_transform(X_elliptic)

# Create Nodes DataFrame
nodes_elliptic = pd.DataFrame(X_elliptic_scaled, columns=X_elliptic.columns)
nodes_elliptic['tx_id'] = df_elliptic['txId']
nodes_elliptic['label'] = y_elliptic.values

# Create Edges DataFrame
edges_elliptic = df_elliptic_edges.copy()
edges_elliptic.columns = ['source','target']

# Final cleanup
import gc
del df_elliptic_features
del df_elliptic_classes
del df_elliptic_edges
del df_elliptic
del X_elliptic
del X_elliptic_scaled
gc.collect()

print(f"Nodes shape: {nodes_elliptic.shape}")
print(f"Edges shape: {edges_elliptic.shape}")

Nodes shape: (46564, 168)
Edges shape: (234355, 2)


## Save Node & Edge CSVs


In [16]:
# Ensure processed directory exists
os.makedirs("./data/processed", exist_ok=True)


# PaySim
nodes_paysim.to_csv("./data/processed/paysim_nodes.csv", index=False)
edges_paysim.to_csv("./data/processed/paysim_edges.csv", index=False)
print("Saved PaySim nodes and edges CSVs.")


# IEEE-CIS

# Train nodes and edges
nodes_ieee_train.to_csv("./data/processed/ieee_train_nodes.csv", index=False)
edges_ieee_train.to_csv("./data/processed/ieee_train_edges.csv", index=False)
print("Saved IEEE train nodes and edges CSVs.")

# Test nodes and edges
nodes_ieee_test.to_csv("./data/processed/ieee_test_nodes.csv", index=False)
edges_ieee_test.to_csv("./data/processed/ieee_test_edges.csv", index=False)
print("Saved IEEE test nodes and edges CSVs.")


# Elliptic
nodes_elliptic.to_csv("./data/processed/elliptic_nodes.csv", index=False)
edges_elliptic.to_csv("./data/processed/elliptic_edges.csv", index=False)
print("Saved Elliptic nodes and edges CSVs.")

print("\nAll node and edge CSVs saved successfully for GNN training and reproducibility.")

Saved PaySim nodes and edges CSVs.
Saved IEEE train nodes and edges CSVs.
Saved IEEE test nodes and edges CSVs.
Saved Elliptic nodes and edges CSVs.

All node and edge CSVs saved successfully for GNN training and reproducibility.


## Conclusion

In this preprocessing pipeline, we transformed our three diverse financial transaction datasets into formats suitable for both graph-based learning and traditional machine learning. Through careful handling of missing values, normalization, and feature encoding, we prepared tabular feature representations for each transaction. Additionally, we explicitly constructed transaction graphs by defining nodes and edges according to relational and temporal dependencies inherent in each dataset. The resulting node and edge CSV files, along with tabular feature arrays, provide a reproducible foundation for downstream modeling, enabling rigorous evaluation of both graph-based and non-graph-based approaches to fraud detection.

## Use for Baseline Machine Learning Models

The node CSVs, containing normalized features and labels, can be used directly as tabular datasets for baseline machine learning models such as Logistic Regression, K-Nearest Neighbors (KNN), Decision Trees, or Random Forests. Each row represents a transaction with its relevant features, while the label column indicates fraud status. These datasets allow quick training, evaluation, and benchmarking of classical models without requiring any graph-specific structure. Baseline models will be useful for establishing performance baselines, feature importance insights, and understanding which individual transaction features are most predictive of fraud.

## Use for Graph Neural Network Models

The combination of nodes CSVs and edges CSVs provides a complete graph representation suitable for Graph Neural Networks (GNNs) such as GCN, GAT, and GraphSAGE. Nodes correspond to individual transactions with associated features, and edges encode relationships between transactions, capturing account sharing, temporal flows, or fund propagation patterns. These files enable the creation of data structures compatible with PyTorch Geometric, DGL, or similar frameworks, allowing models to learn from both transaction attributes and the underlying network topology. By leveraging connectivity information, GNNs can detect higher-order patterns, such as repeated or coordinated fraudulent behaviors, which baseline tabular models may fail to capture.